In [1]:
import os
import pandas as pd
import cv2
import torch
from torch.utils.data import Dataset, random_split
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
from PIL import Image

/usr/local/lib/python3.10/dist-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 1.4.23 (you have 1.4.15). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [2]:
class AugmentedECGDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels  # DataFrame containing labels and corresponding IDs
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Extract the image path and ID
        image_path = self.image_paths[idx]
        image_id = os.path.basename(image_path).split('_')[1].split('.')[0]  # Extract ID from path (e.g., '000100')

        # Find the corresponding row in the labels DataFrame using the ID
        label_row = self.labels[self.labels['ID'] == int(image_id)]  # Assuming 'ID' column is integers

        if label_row.empty:
            raise ValueError(f"ID {image_id} not found in the labels DataFrame.")

        # Extract the label values (assuming they are in the other columns of the DataFrame)
        labels = label_row.iloc[0, 1:].values  # Skipping the ID column

        # Try to load the image
        try:
            image = Image.open(image_path).convert("RGB")
        except (IOError, OSError) as e:
            print(f"Error loading image {image_path}: {e}")
            return None  # Skip this image if it cannot be loaded

        # Convert image to numpy array
        image = np.array(image)

        # Apply transformations (ensure the transformation works with named arguments)
        if self.transform:
            augmented = self.transform(image=image)  # Use 'image=image' to pass it as a named argument
            image = augmented['image']

        return {"pixel_values": image, "labels": torch.tensor(labels, dtype=torch.float32)}

In [3]:
# Custom Resize with Anti-Aliasing
class ResizeWithAntiAliasing(A.ImageOnlyTransform):
    def __init__(self, width, height, always_apply=False, p=1.0):
        super(ResizeWithAntiAliasing, self).__init__(always_apply, p)
        self.width = width
        self.height = height

    def apply(self, image, **params):
        # Convert the image to a PIL image, apply resizing with anti-aliasing
        image_pil = Image.fromarray(image)
        image_resized = image_pil.resize((self.width, self.height), Image.Resampling.LANCZOS)
        return np.array(image_resized)

In [4]:
# Define augmentations for training
train_transform = A.Compose([
    ResizeWithAntiAliasing(width=224, height=224),  # Resize to 224x224
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=15, p=0.7),  # Apply shift, scale, rotate
    A.ToGray(p=1.0),  # Convert to grayscale
    A.GaussianBlur(blur_limit=(3, 7), p=1.0),  # Apply Gaussian blur
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),  # Normalize
    ToTensorV2(),  # Convert to PyTorch tensor
])

In [5]:
# Define transformations for validation (no augmentation)
val_transform = A.Compose([
    ResizeWithAntiAliasing(width=224, height=224),  # Resize to 224x224
    A.ToGray(p=1.0),  # Convert to grayscale
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

In [6]:
# Load the CSV file with labels
labels_df = pd.read_csv("/kaggle/input/bhf-data-science-centre-ecg-challenge/train_final.csv")

In [7]:
with open('/kaggle/input/bhf-reference-files/valid_images_list.txt', 'r') as file:
    valid_image_paths = [line.strip() for line in file]

with open('/kaggle/input/bhf-reference-files/valid_test_images_list.txt', 'r') as file:
    valid_test_images = [line.strip() for line in file]

In [8]:
# Create the datasets A training and validation
train_dataset = AugmentedECGDataset(valid_image_paths, labels_df, transform=train_transform)
val_dataset = AugmentedECGDataset(valid_test_images, labels_df, transform=val_transform)

In [9]:
from torch.utils.data import DataLoader

batch_size = 10

# Create DataLoaders for training and validation datasets
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

In [10]:
print("Dataset read in!")

Dataset read in!


In [11]:
from torchvision import models
import torch.nn as nn
import torch.optim as opt

# Load pretrained ResNet model
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Modify the final layer for 5 output labels (multi-label classification)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 5)  # Output layer for 5 labels

# Loss function for multi-label classification
criterion = nn.BCEWithLogitsLoss()  # Suitable for multi-label classification

# Optimizer setup
learning_rate = 0.001
optimizer = opt.Adam(
    model.parameters(),  # Parameters of the model you want to optimize
    lr=learning_rate,    # Learning rate
    betas=(0.9, 0.999),  # Beta values for the first and second moment estimates (default values are usually fine)
    eps=1e-8,            # A small constant added to improve numerical stability (default is fine)
    weight_decay=0.001   # L2 regularization (set to 0 to disable weight decay)
)



Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 222MB/s]


In [12]:
# Move model to the appropriate device (CUDA, MPS, or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_built() else "cpu")

print(device)

# Move the model to the selected device
model = model.to(device)

cuda


In [13]:
print('Beginning model training')

Beginning model training


In [14]:

num_epochs = 5
training_losses = []
validation_accuracies = []

for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    total_loss = 0.0

    for batch in train_dataloader:
        # Move batch data to GPU
        inputs = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Training Loss: {total_loss / len(train_dataloader)}")
    training_losses.append(total_loss/len(train_dataloader))
    # Evaluate on validation data after every epoch
    model.eval()
    correct_predictions = 0
    total_predictions = 0

    with torch.no_grad():
        for batch in val_dataloader:
            inputs = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            # Forward pass
            outputs = model(inputs)

            # Apply sigmoid to get probabilities (since the output layer uses Sigmoid for multi-label classification)
            probabilities = torch.sigmoid(outputs)

            # Convert probabilities to binary predictions (threshold at 0.5)
            predictions = (probabilities > 0.5).float()

            # Count correct predictions
            correct_predictions += (predictions == labels).sum().item()
            total_predictions += labels.numel()  # Total number of labels in the batch

    # Calculate accuracy
    accuracy = correct_predictions / total_predictions
    print(f"Validation Accuracy: {accuracy:.4f}")
    validation_accuracies.append(accuracy)

Epoch 1/5, Training Loss: 0.46590845191542263
Validation Accuracy: 0.8195
Epoch 2/5, Training Loss: 0.45385194812553237
Validation Accuracy: 0.8195
Epoch 3/5, Training Loss: 0.45305277857877346
Validation Accuracy: 0.8195
Epoch 4/5, Training Loss: 0.4521740163111194
Validation Accuracy: 0.8195
Epoch 5/5, Training Loss: 0.4522275220823637
Validation Accuracy: 0.8195


In [15]:
training_loss_arr = np.array(training_losses)
validation_acc_arr = np.array(validation_accuracies)

In [16]:
np.savetxt('training_loss.txt', training_loss_arr, fmt='%d')

In [17]:
np.savetxt('validation_accuracy.txt', validation_acc_arr, fmt='%d')

In [18]:
torch.save(model.state_dict(), 'final_model')